# Imports

In [28]:
import os
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timezone, timedelta
import pprint

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, fbeta_score, average_precision_score, roc_auc_score

import gc


In [14]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd

/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [16]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
exp_manager = ExperimentManager()

# Configuration

In [17]:
CSV_PATH = "/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv"
BASE_OUTPUT_DIR = "./dataset_processed"
TIME_WINDOW_SEC = 30  # Each graph represents 30 seconds
CHUNK_SIZE = 1000000   # Rows to read at a time in RAM

In [18]:
cicids2018v3_wed2802_orig = pd.read_csv(CSV_PATH)
cicids2018v3_wed2802 = cicids2018v3_wed2802_orig.copy()
# Convert 'FLOW_START_TIME' to datetime type after loading
cicids2018v3_wed2802['FLOW_START_TIME'] = pd.to_datetime(cicids2018v3_wed2802['FLOW_START_TIME'])

timestamps = cicids2018v3_wed2802['FLOW_START_TIME']
GLOBAL_START_TIME = timestamps.min()
GLOBAL_END_TIME = timestamps.max()
TOTAL_DURATION = GLOBAL_END_TIME - GLOBAL_START_TIME

# Limits for Train/Val/Test
LIMIT_TRAIN = GLOBAL_START_TIME + (TOTAL_DURATION * 0.70)
LIMIT_VAL = GLOBAL_START_TIME + (TOTAL_DURATION * 0.85)

print(f"Global start time: {GLOBAL_START_TIME}")
print(f"Global end time: {GLOBAL_END_TIME}")

print(f"Limit train: {LIMIT_TRAIN}")
print(f"Limit val: {LIMIT_VAL}")

Global start time: 2018-02-28 00:12:55.854000
Global end time: 2018-02-28 23:59:59.321000
Limit train: 2018-02-28 16:51:52.280900
Limit val: 2018-02-28 20:25:55.800950


In [19]:
NUMERIC_FEATURES = [
    'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT',
    'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG',
    'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_STDDEV',
    'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
    'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_PKTS',
    'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
    'TCP_FLAGS', 'MIN_TTL', 'MAX_TTL'
]

CATEGORICAL_FEATURES = [
    'PROTOCOL', 'L4_DST_PORT'
]

# Functions

In [20]:
def apply_port_roles(df):
    """Create manual One-Hot columns for port roles"""
    # 1. Web
    df['Role_Web'] = df['L4_DST_PORT'].isin([80, 443, 8080, 8443, 81, 3128, 8545]).astype(int)
    # 2. Admin
    df['Role_Admin'] = df['L4_DST_PORT'].isin([22, 222, 2222, 23, 2323, 3389, 3390, 3394, 5900, 5901, 5555, 21, 2131]).astype(int)
    # 3. Windows
    df['Role_Win'] = df['L4_DST_PORT'].isin([445, 135, 137, 138, 139]).astype(int)
    # 4. DNS/Infra
    df['Role_Infra'] = df['L4_DST_PORT'].isin([53, 5355, 67, 547, 123, 1900, 5060]).astype(int)
    # 5. DB
    df['Role_DB'] = df['L4_DST_PORT'].isin([1433, 3306, 5432, 6379, 27017]).astype(int)
    # 6. Privileged Low
    df['Role_LowPriv'] = ((df['L4_DST_PORT'] < 1024) & (df['Role_Web']==0) & (df['Role_Admin']==0) & (df['Role_Win']==0) & (df['Role_Infra']==0)).astype(int)
    # 7. Privileged High
    df['Role_HighPriv'] = (df['L4_DST_PORT'] >= 1024).astype(int)

    return df

In [21]:
def process_chunk(chunk):
    # Clean 0.0.0.0
    chunk = chunk[ (chunk['IPV4_SRC_ADDR'] != '0.0.0.0') & (chunk['IPV4_DST_ADDR'] != '0.0.0.0') ].copy()
    chunk['FLOW_START_TIME'] = pd.to_datetime(chunk['FLOW_START_TIME'])

    # Log1p numeric features
    chunk[NUMERIC_FEATURES] = np.log1p(chunk[NUMERIC_FEATURES].astype(float))

    # One-Hot port
    chunk = apply_port_roles(chunk)

    # One-hot protocol
    chunk['Proto_TCP'] = (chunk['PROTOCOL'] == 6).astype(int)
    chunk['Proto_UDP'] = (chunk['PROTOCOL'] == 17).astype(int)
    chunk['Proto_ICMP_ICMPv6'] = ((chunk['PROTOCOL'] == 1) | (chunk['PROTOCOL'] == 58)).astype(int)
    chunk['Proto_IGMP'] = (chunk['PROTOCOL'] == 2).astype(int)
    chunk['Proto_Other'] = ((chunk['PROTOCOL'] != 6) & (chunk['PROTOCOL'] != 17) & (chunk['PROTOCOL'] != 1) & (chunk['PROTOCOL'] != 58) & (chunk['PROTOCOL'] != 2)).astype(int)

    # Target
    chunk['Target'] = chunk['Attack'].apply(lambda x: 1 if x == 'Infilteration' else 0)

    return chunk

In [22]:
def calculate_metrics(y_true, y_probs, probs_threshold=0.5):
    preds = (y_probs > probs_threshold).astype(int)

    # Calculate standard metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)

    # Calculate other metrics
    f2 = fbeta_score(y_val, preds, beta=2)
    ap = average_precision_score(y_val, y_probs)
    roc = roc_auc_score(y_val, y_probs)

    # Calculate False Positive Rate (FPR)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": tp, "FP": fp, "TN": tn, "FN": fn, "Total_Flows": len(y_true)
    }

# Load and split

In [24]:
train_list = []
val_list = []
test_list = []

for i, chunk in enumerate(pd.read_csv(CSV_PATH, usecols=NUMERIC_FEATURES + ['L4_DST_PORT', 'PROTOCOL', 'Attack', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'FLOW_START_TIME'], chunksize=CHUNK_SIZE)):

    df_proc = process_chunk(chunk)

    # Split by time
    mask_train = df_proc['FLOW_START_TIME'] < LIMIT_TRAIN
    mask_val = (df_proc['FLOW_START_TIME'] >= LIMIT_TRAIN) & (df_proc['FLOW_START_TIME'] < LIMIT_VAL)
    mask_test = df_proc['FLOW_START_TIME'] >= LIMIT_VAL

    if mask_train.sum() > 0:
        train_list.append(df_proc[mask_train])

    if mask_val.sum() > 0:
        val_list.append(df_proc[mask_val])

    if mask_test.sum() > 0:
        test_list.append(df_proc[mask_test])

    print(f"Chunk {i} processed. Train rows: {mask_train.sum()}, Val rows: {mask_val.sum()}, Test rows: {mask_test.sum()}")

# Concatenate
df_train = pd.concat(train_list, ignore_index=True)
df_val = pd.concat(val_list, ignore_index=True)
df_test = pd.concat(test_list, ignore_index=True)

df_train.to_parquet(os.path.join(BASE_OUTPUT_DIR, "train_tabular.parquet"))
df_val.to_parquet(os.path.join(BASE_OUTPUT_DIR, "val_tabular.parquet"))
df_test.to_parquet(os.path.join(BASE_OUTPUT_DIR, "test_tabular.parquet"))


del train_list, val_list, test_list
gc.collect()

# Columns (numeric and one-hot)
FEATS = NUMERIC_FEATURES + ['Role_Web', 'Role_Admin', 'Role_Win', 'Role_Infra', 'Role_DB', 'Role_LowPriv', 'Role_HighPriv', 'Proto_TCP', 'Proto_UDP', 'Proto_ICMP_ICMPv6', 'Proto_IGMP', 'Proto_Other']

X_train = df_train[FEATS]
y_train = df_train['Target']
X_val = df_val[FEATS]
y_val = df_val['Target']
X_test = df_test[FEATS]
y_test = df_test['Target']

print(f"Final dimension: Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")
print(f"Attacks in Train: {y_train.sum()}")
print(f"Attacks in Val: {y_val.sum()}")
print(f"Attacks in Test: {y_test.sum()}")

Chunk 0 processed. Train rows: 994926, Val rows: 0, Test rows: 0
Chunk 1 processed. Train rows: 46719, Val rows: 788681, Test rows: 159306
Chunk 2 processed. Train rows: 0, Val rows: 0, Test rows: 32375
Final dimension: Train (1041645, 32), Val (788681, 32), Test (191681, 32)
Attacks in Train: 49564
Attacks in Val: 31329
Attacks in Test: 3434


# Training and testing

In [25]:
model_config = {
    "model_name": "RandomForest",
    "type": "Baseline_RF",
    "model_params": {
        "n_estimators": 100,
        "class_weight": "balanced",
        "max_depth": 15,
        "n_jobs": -1,
        "random_state": 42
    },
    "prob_threshold": 0.5
}

rf = RandomForestClassifier(**model_config["model_params"])

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=15, n_jobs=-1,
                       random_state=42)

In [27]:
# Feature Importance
importances = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
print("\n--- Top 10 Features ---")
print(importances.head(10))


--- Top 10 Features ---
OUT_BYTES          0.098944
IN_BYTES           0.097024
TCP_FLAGS          0.093019
MIN_IP_PKT_LEN     0.084104
MAX_IP_PKT_LEN     0.075419
TCP_WIN_MAX_IN     0.057463
OUT_PKTS           0.053690
TCP_WIN_MAX_OUT    0.051204
IN_PKTS            0.040140
MIN_TTL            0.037286
dtype: float64


In [30]:
# Testing (VAL)
print("\nTesting in Validation...")
preds = rf.predict(X_val) # this assumes prob_threshold=0.5
y_probs_rf = rf.predict_proba(X_val)[:, 1]

print("--- Classification Report ---")
print(classification_report(y_val, preds))

print("-- Confusion Matrix ---")
print(confusion_matrix(y_val, preds))

metrics_rf = calculate_metrics(y_val, y_probs_rf)
metrics_rf = {
    k: float(v) if isinstance(v, (np.floating, np.integer)) else v
    for k, v in metrics_rf.items()
}
pprint.pprint(metrics_rf)


# Register and save
exp_manager.log_experiment(model_config=model_config, metrics=metrics_rf, model_object=rf)


Testing in Validation...
--- Classification Report ---
              precision    recall  f1-score   support

           0       0.99      0.68      0.81    757352
           1       0.10      0.86      0.18     31329

    accuracy                           0.69    788681
   macro avg       0.55      0.77      0.49    788681
weighted avg       0.96      0.69      0.78    788681

-- Confusion Matrix ---
[[515438 241914]
 [  4414  26915]]
{'AUC-PR': 0.2861065661669485,
 'AUC-ROC': 0.8536069263173388,
 'F1': 0.1793388815223982,
 'F2': 0.34143525859772417,
 'FN': 4414.0,
 'FP': 241914.0,
 'FPR': 0.3194208241346164,
 'Precision': 0.10011940676043135,
 'Recall': 0.8591081745347761,
 'TN': 515438.0,
 'TP': 26915.0,
 'Total_Flows': 788681}
Experiment recorded in ./logs/experiments_log.csv
Saved model: RandomForest_20260126_1653_AUC-PR_0.2861


In [36]:
y_probs = rf.predict_proba(X_val)[:, 1]


best_th = 0.5
best_f1 = 0.0


thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

for th in thresholds:
    preds_th = (y_probs >= th).astype(int)

    f1 = f1_score(y_val, preds_th)
    prec = precision_score(y_val, preds_th)
    rec = recall_score(y_val, preds_th)

    print(f"Umbral {th:.2f} -> F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_th = th

print(f"\nBest configuration Baseline: Umbral {best_th} con F1 {best_f1:.4f}")

Umbral 0.50 -> F1: 0.1793 | Precision: 0.1001 | Recall: 0.8591
Umbral 0.60 -> F1: 0.1979 | Precision: 0.1146 | Recall: 0.7250
Umbral 0.70 -> F1: 0.3130 | Precision: 0.3236 | Recall: 0.3032
Umbral 0.80 -> F1: 0.3368 | Precision: 0.4854 | Recall: 0.2578
Umbral 0.90 -> F1: 0.3282 | Precision: 0.5842 | Recall: 0.2282
Umbral 0.95 -> F1: 0.3049 | Precision: 0.5900 | Recall: 0.2056

Best configuration Baseline: Umbral 0.8 con F1 0.3368
